In [ ]:
%load_ext autoreload
%autoreload 2

In [43]:
import pandas as pd
import numpy as np
import os
import sys
from typing import List

sys.path.append('/Users/harendrakumar/Documents/bank_loan_prediction/')

In [2]:
!ls /Users/harendrakumar/Documents/bank_loan_prediction/ML_Pipelines/

data_folder ml


In [8]:
from ML_Pipelines.ml.utils.data_loader import get_latest_partition_data
from ML_Pipelines.ml.pipelines.ingestion import IngestionPipeline
from ML_Pipelines.ml.utils import databalancer, data_loader, utility

In [9]:
filename = get_latest_partition_data('../../data_folder/train')
filename

'credit_train_20241027.csv'

In [30]:
df = IngestionPipeline(data_dir="../../data_folder/train").load_data()

INFO:ML_Pipelines.ml.pipelines.ingestion:Latest partition file ../../data_folder/train/credit_train_20241027.csv loaded successfully.


In [31]:
df.head()

,Loan ID,Customer ID,Loan Status,Current Loan Amount,Term,Credit Score,Annual Income,Years in current job,Home Ownership,Purpose,Monthly Debt,Years of Credit History,Months since last delinquent,Number of Open Accounts,Number of Credit Problems,Current Credit Balance,Maximum Open Credit,Bankruptcies,Tax Liens
0,14dd8831-6af5-400b-83ec-68e61888a048,981165ec-3274-42f5-a3b4-d104041a9ca9,Fully Paid,445412.0,Short Term,709.0,1167493.0,8 years,Home Mortgage,Home Improvements,5214.74,17.2,NaN,6.0,1.0,228190.0,416746.0,1.0,0.0
1,4771cc26-131a-45db-b5aa-537ea4ba5342,2de017a3-2e01-49cb-a581-08169e83be29,Fully Paid,262328.0,Short Term,NaN,NaN,10+ years,Home Mortgage,Debt Consolidation,33295.98,21.1,8.0,35.0,0.0,229976.0,850784.0,0.0,0.0
2,4eed4e6a-aa2f-4c91-8651-ce984ee8fb26,5efb2b2b-bf11-4dfd-a572-3761a2694725,Fully Paid,99999999.0,Short Term,741.0,2231892.0,8 years,Own Home,Debt Consolidation,29200.53,14.9,29.0,18.0,1.0,297996.0,750090.0,0.0,0.0
3,77598f7b-32e7-4e3b-a6e5-06ba0d98fe8a,e777faab-98ae-45af-9a86-7ce5b33b1011,Fully Paid,347666.0,Long Term,721.0,806949.0,3 years,Own Home,Debt Consolidation,8741.90,12.0,NaN,9.0,0.0,256329.0,386958.0,0.0,0.0
4,d4062e70-befa-4995-8643-a0de73938182,81536ad9-5ccf-4eb8-befb-47a4d608658e,Fully Paid,176220.0,Short Term,NaN,NaN,5 years,Rent,Debt Consolidation,20639.70,6.1,NaN,15.0,0.0,253460.0,427174.0,0.0,0.0


In [32]:
df.columns

Index(['Loan ID', 'Customer ID', 'Loan Status', 'Current Loan Amount', 'Term',
       'Credit Score', 'Annual Income', 'Years in current job',
       'Home Ownership', 'Purpose', 'Monthly Debt', 'Years of Credit History',
       'Months since last delinquent', 'Number of Open Accounts',
       'Number of Credit Problems', 'Current Credit Balance',
       'Maximum Open Credit', 'Bankruptcies', 'Tax Liens'],
      dtype='object')

In [33]:
df.shape

(100000, 19)

In [34]:
df.drop(["Loan ID", "Customer ID"], axis=1, inplace=True)

In [35]:
cat_columns = df.select_dtypes(include='object').columns
cat_columns

Index(['Loan Status', 'Term', 'Years in current job', 'Home Ownership',
       'Purpose'],
      dtype='object')

In [26]:
df['Loan Status'] = df['Loan Status'].astype('category')

mapping  = dict(enumerate(df['Loan Status'].cat.categories))

df['Loan Status'] = df['Loan Status'].cat.codes

In [28]:
def encode_categorical_columns(df: pd.DataFrame, categorical_cols: List):
    for col in categorical_cols:
        df[col] = df[col].astype('category')
        mapping = dict(enumerate(df[col].cat.categories))
        print(mapping)

        df[col] = df[col].cat.codes

    return df

In [36]:
df = encode_categorical_columns(df=df, categorical_cols=cat_columns)

{0: 'Charged Off', 1: 'Fully Paid'}
{0: 'Long Term', 1: 'Short Term'}
{0: '1 year', 1: '10+ years', 2: '2 years', 3: '3 years', 4: '4 years', 5: '5 years', 6: '6 years', 7: '7 years', 8: '8 years', 9: '9 years', 10: '< 1 year'}
{0: 'HaveMortgage', 1: 'Home Mortgage', 2: 'Own Home', 3: 'Rent'}
{0: 'Business Loan', 1: 'Buy House', 2: 'Buy a Car', 3: 'Debt Consolidation', 4: 'Educational Expenses', 5: 'Home Improvements', 6: 'Medical Bills', 7: 'Other', 8: 'Take a Trip', 9: 'major_purchase', 10: 'moving', 11: 'other', 12: 'renewable_energy', 13: 'small_business', 14: 'vacation', 15: 'wedding'}


In [37]:
df['Loan Status'].unique()

array([1, 0], dtype=int8)

In [38]:
df['Term'].unique()

array([1, 0], dtype=int8)

In [44]:
df['Years in current job'] = np.where(df['Years in current job'] == -1, 11, df['Years in current job'])

In [46]:
df[df['Years in current job'] == 11].head(2)

,Loan Status,Current Loan Amount,Term,Credit Score,Annual Income,Years in current job,Home Ownership,Purpose,Monthly Debt,Years of Credit History,Months since last delinquent,Number of Open Accounts,Number of Credit Problems,Current Credit Balance,Maximum Open Credit,Bankruptcies,Tax Liens
29,1,107404.0,1,NaN,NaN,11,1,11,19238.07,43.7,NaN,5.0,0.0,28956.0,58014.0,0.0,0.0
73,1,311058.0,0,675.0,1343167.0,11,1,3,21378.80,31.4,17.0,11.0,0.0,247912.0,541596.0,0.0,0.0


In [41]:
sorted(df['Years in current job'].unique())

[-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

In [23]:
df['Home Ownership'].unique()

array(['Home Mortgage', 'Own Home', 'Rent', 'HaveMortgage'], dtype=object)

In [24]:
df['Purpose'].unique()

array(['Home Improvements', 'Debt Consolidation', 'Buy House', 'other',
       'Business Loan', 'Buy a Car', 'major_purchase', 'Take a Trip',
       'Other', 'small_business', 'Medical Bills', 'wedding', 'vacation',
       'Educational Expenses', 'moving', 'renewable_energy'], dtype=object)

In [15]:
# remove ID columns as it is not needed

def drop_unwanted_columns(df: pd.DataFrame, col_list: List) -> None:
    print(f"Dropping columns: {col_list}")
    return df.drop(col_list, axis=1)
    

In [ ]:
new_df = drop_unwanted_columns(df=df, col_list=["Loan ID", "Customer ID"])

In [ ]:
new_df.head(4)

In [ ]:
new_df.shape

In [ ]:
(new_df.isnull().sum() / new_df.shape[0]) * 100

In [ ]:
new_df["Loan Status"].unique()

In [ ]:
new_df["Term"].unique()

In [ ]:
for col in df.columns:
    print(f"{col}: {df[col].dtypes}")

In [ ]:
categorical_columns = new_df.select_dtypes(include="object").columns.to_list()
categorical_columns

In [ ]:
new_df["Loan Status"].isnull().sum()

In [ ]:
def drop_columns_having_nulls_above_threshold(df, threshold=50):
    null_percent = df.isnull().mean() * 100
    cols_to_drop = null_percent[null_percent > threshold].index.tolist()
    print(f"dropping columns: {','.join(col for col in cols_to_drop)}")
    return df.drop(columns=cols_to_drop).dropna(how='all')

In [ ]:
drop_columns_having_nulls_above_threshold(df=new_df).head(5)

In [ ]:
cat_cols = new_df.select_dtypes(include=['object']).columns
cat_cols

In [ ]:
new_df['Loan Status'].mode()[0]

In [ ]:
def fill_categorical_values(data: pd.DataFrame) ->pd.DataFrame:
    """
      Args:
        data: pd.DataFrame
      Returns:
          pd.DataFrame
    """
    cat_cols = data.select_dtypes(include=['object']).columns
    for col in cat_cols:
        data[col] = data[col].fillna(data[col].mode()[0])
    return data

In [ ]:
new_df = fill_categorical_values(data=new_df)

In [ ]:
def fill_numeric_null_values(data: pd.DataFrame) ->pd.DataFrame:
    """
    Args:
        data pd.DataFrame: pandas dataframe
    Returns:
        pd.DataFrame
    """
    num_cols = data.select_dtypes(include=['number']).columns
    for col in num_cols:
        data[col] = data[col].fillna(data[col].mean())
    return data

In [ ]:
new_df = fill_numeric_null_values(data=new_df)

In [ ]:
new_df.isnull().sum()

In [ ]:
BASE_DIR = "../../data_folder/train"
FILENAME = filename

In [ ]:
file_path = os.path.join(BASE_DIR, FILENAME)

In [ ]:
file_path.split("/")[-1].split(".")[-1]